# Debugging With an AI Assistant

Below is a small classifier and training loop for a synthetic binary classification dataset. It was put together with the help of an AI assistant, and it has **two bugs**, seen one at a time.

For each bug:
1. Run the cell. Read whatever it tells you (a traceback, or a failed assertion).
2. Investigate -- with the assistant if you like, in the notebook if you prefer. If you use the assistant, pause after every hypothesis it offers and judge whether it's plausible *before* copying it into notebook and executing it.
3. Before you touch the fix cell: write two sentences, in your own words, explaining the root cause.
4. Fix the code and re-run to confirm.

---

## Setup

A small synthetic binary classification dataset. Nothing to download and runs in seconds on CPU.

**Note**: You may need to install scikit-learn for this code to work.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

device = 'cpu'
torch.manual_seed(0)

X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=15,
    n_redundant=2, random_state=0,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y,
)

X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
y_test_t = torch.tensor(y_test, dtype=torch.float32, device=device)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True,
)

print(X_train_t.shape, X_test_t.shape)

torch.Size([800, 20]) torch.Size([200, 20])


A small helper to evaluate accuracy on held-out data -- you won't need to change this.

In [4]:
def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        logits = model(X).squeeze(1)
        preds = (torch.sigmoid(logits) > 0.5).float()
        accuracy = (preds == y).float().mean().item()
    model.train()
    return accuracy

---

## Bug 1: a crash

The model below is meant to classify the dataset defined above. Run the next cell.

In [5]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=X_train_t.shape[1])
logits = model(X_train_t)
print(logits.shape)

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)

**Root cause:** *Write two sentence about your analysis. Change the code below to fix the crash.*

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=X_train_t.shape[1])
logits = model(X_train_t)
print(logits.shape)  # torch.Size([800, 1])

---

## Bug 2: no crash, but something is not correct

With the model fixed, here is a training loop. Run it, then check the assertion in the cell after it. Nothing raises an exception during training. The failure only shows up in the final check. This is the harder kind of bug: the code runs, looks reasonable, and still doesn't do what it should.

In [ ]:
def train_model(model, loader, epochs=50, lr=0.05):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for xb, yb in loader:
            logits = model(xb).squeeze(1)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(loader))
    return losses

model = MLP(input_dim=X_train_t.shape[1])
losses = train_model(model, train_loader)
print(f"first epoch loss: {losses[0]:.3f}, last epoch loss: {losses[-1]:.3f}")

test_acc = evaluate(model, X_test_t, y_test_t)
print(f"test accuracy: {test_acc:.3f}")
assert test_acc > 0.8, f"Test accuracy too low: {test_acc:.3f}. Something is still wrong."


**Root cause:** *Write two sentence about your analysis. Change the code below to fix the crash.*

In [ ]:
def train_model(model, loader, epochs=50, lr=0.05):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for xb, yb in loader:
            logits = model(xb).squeeze(1)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(loader))
    return losses

model = MLP(input_dim=X_train_t.shape[1])
losses = train_model(model, train_loader)
print(f"first epoch loss: {losses[0]:.3f}, last epoch loss: {losses[-1]:.3f}")

test_acc = evaluate(model, X_test_t, y_test_t)
print(f"test accuracy: {test_acc:.3f}")
assert test_acc > 0.8, f"Test accuracy too low: {test_acc:.3f}. Something is still wrong."


---

## Conclusions

We saw two very different bugs:

- **Bug 1** expressed itself immediately with a traceback, so finding the bug was easy. The challenge was reading the shapes in the error message.
- **Bug 2** never raised any exception at all. The only signal was a failed assertion on the final outcome, which meant investigating the *behaviour* of the training loop rather than its syntax. These bugs will be more common and require either additional test cases or deeper analysis to be identified.